# 🌊 MambaMPD Geospatial Marine Oil Spill Pipeline on Kaggle

This notebook demonstrates the end-to-end geospatial workflow:
1. **Load Sentinel-1 GeoTIFF / SAR rasters** preserving CRS and geospatial transform.
2. **Load your trained MambaMPD checkpoint** without altering model architecture.
3. **Apply exact M4D training preprocessing** (divisible-by-32 reflection padding & normalization).
4. **Generate 5-class predictions** and extract **Class 1 (Oil Spill)**.
5. **Vectorize oil spills into georeferenced polygons**.
6. **Calculate geodesic area in km²**.
7. **Export standard RFC 7946 GeoJSON**.

In [ ]:
# Step 1: Install required dependencies in Kaggle
!pip install -q rasterio shapely pyproj geopandas
!pip install -q monai "segmentation-models-pytorch>=0.3.3" PyWavelets einops timm

In [ ]:
# Step 2: Auto-detect Kaggle inputs and model checkpoint
import os, sys, glob
sys.path.append('.')

from geospatial import (
    load_sentinel1_raster,
    vectorize_oil_spill,
    export_preview_png,
    MambaMPDInferenceEngine,
)

CHECKPOINT_PATH = "/kaggle/input/mambampd-checkpoints/mambampd_best.pt" # Or local mambampd_best.pt
if not os.path.isfile(CHECKPOINT_PATH):
    candidates = glob.glob("/kaggle/input/**/*.pt", recursive=True) + glob.glob("**/*.pt", recursive=True)
    if candidates:
        CHECKPOINT_PATH = candidates[0]
print(f"Using MambaMPD Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# Step 3: Run Batch Pipeline over dataset
!python kaggle/run_pipeline_kaggle.py \
    --input_path /kaggle/input/m4d-oil-spill-detection-dataset/test/images \
    --checkpoint {CHECKPOINT_PATH} \
    --output_dir /kaggle/working/detections \
    --oil_threshold 0.5 \
    --min_pixel_area 5

In [ ]:
# Step 4: Display Summary Results Table
import pandas as pd
summary_csv = "/kaggle/working/detections/oil_spill_summary.csv"
if os.path.isfile(summary_csv):
    df = pd.read_csv(summary_csv)
    print(f"Total Images Processed: {len(df)}")
    print(f"Total Oil Spills: {df['total_spills_detected'].sum()}")
    print(f"Total Area: {df['total_area_km2'].sum():.4f} km²")
    display(df.head(10))
else:
    print("Run Step 3 to generate summary.")

In [ ]:
# Step 5: (Optional) Launch FastAPI Backend Server from Kaggle
# You can run: uvicorn server.main:app --host 0.0.0.0 --port 8000